# client

> Client for interacting with the Fewsats API

In [ ]:
#| default_exp core

In [ ]:
#| export
from fastcore.utils import *
import os
import httpx
from typing import Dict, Any, List

In [ ]:
#| hide 
from dotenv import load_dotenv
from fastcore.test import *

In [ ]:
#| hide
load_dotenv()

True

The `Client` class handles authentication and provides the foundation for our API interactions.

In [ ]:
#| export
class Client:
    "Client for interacting with the Fewsats API"
    def __init__(self,
                 api_key: str = None, # The API key for the Fewsats account
                 base_url: str = "https://hub-5n97k.ondigitalocean.app"): # The Fewsats API base URL
        self.api_key = api_key or os.environ.get("FEWSATS_API_KEY")
        if not self.api_key:
            raise ValueError("The api_key client option must be set either by passing api_key to the client or by setting the FEWSATS_API_KEY environment variable")
        self.base_url = base_url
        self._httpx_client = httpx.Client()
        self._httpx_client.headers.update({"Authorization": f"Token {self.api_key}"})


In [ ]:
k = os.getenv("FEWSATS_API_KEY")
fs = Client(api_key=k)

test_eq(fs.api_key, k)
test_eq(fs._httpx_client.headers["Authorization"], f"Token {k}")

## Methods

In [ ]:
#| export
@patch
def _request(self: Client, 
             method: str, # The HTTP method to use
             path: str, # The path to request
             **kwargs) -> Dict[str, Any]:
    "Makes an authenticated request to Fewsats API"
    url = f"{self.base_url}/{path}"
    return  self._httpx_client.request(method, url, **kwargs)

In [ ]:
# r  = fs._request("GET", "v0/stripe/payment-methods")
r  = fs._request("GET", "v0/users/me")
test_eq(r.status_code, 200)

### User Info

In [ ]:
#| export

@patch
def me(self: Client):
    "Retrieve the user's info."
    r = self._request("GET", "v0/users/me")
    r.raise_for_status()
    return r.json()

In [ ]:
fs.me()

{'name': 'Fewsats',
 'last_name': 'Tester',
 'email': 'test@fewsats.com',
 'billing_info': None,
 'id': 15,
 'created_at': '2024-12-18T18:19:00.531Z'}

### Balance 

In [ ]:
#| export 

@patch
def balance(self: Client):
    "Retrieve the balance of the user's wallet."
    r = self._request("GET", "v0/wallets")
    r.raise_for_status()
    return r.json()

In [ ]:
fs.balance()

[{'id': 15, 'balance': 9988, 'currency': 'usd'}]

### Payment Methods

Retrieve the user's payment methods. Useful for checking which card will be used for purchases.

In [ ]:
#| export
@patch
def payment_methods(self: Client) -> List[Dict[str, Any]]:
    "Retrieve the user's payment methods, raises an exception for error status codes."
    r = self._request("GET", "v0/stripe/payment-methods")
    r.raise_for_status()
    return r.json()

In [ ]:
fs = Client()
pm = fs.payment_methods()
pm

[{'id': 5,
  'last4': '4242',
  'brand': 'Visa',
  'exp_month': 12,
  'exp_year': 2034,
  'is_default': True}]

In [ ]:
assert isinstance(pm, list)

### Simulate a Purchase

Simulate a purchase and return the resulting state. Useful, for example, to check if a CC charge is needed or the purchase will use the balance.

In [ ]:
#| export

@patch
def simulate_payment(self: Client,
                    amount: str): # The amount in USD cents
    "Simulates a purchase, raises an exception for error status codes."
    assert amount.isdigit()
    return self._request("POST", "v0/l402/preview/purchase/amount", json={"amount_usd": amount})


In [ ]:
p = fs.simulate_payment(amount="300") # 3.00 USD
p.json()

{'invoice': {'description': 'USD amount preview',
  'amount_usd': 300,
  'amount_btc': 0,
  'macaroon': '',
  'invoice': ''},
 'transaction': {'current_balance': 9988,
  'balance_to_apply': 300,
  'amount_to_charge': 0,
  'final_balance': 9688},
 'already_purchased': False,
 'purchase': None}

In [ ]:
#| hide
test_eq(p.status_code, 200)

The following is an example of how to pay a lightning invoice. This is a low level method that should not used by most users. This method will use the default payment method if a charge is needed.

In [ ]:
#| export

@patch
def _pay_ln(self: Client,
         ln_invoice: str, # The Lightning Network invoice to pay
         description: str, # Short payment description
         l402_url: str = ""): # L402 URL of the resource
     "Pay an invoice, raises an exception for error status codes."
     p = {"invoice": ln_invoice, "description": description, "l402_url": l402_url, "macaroon":""}
     return self._request("POST", "v0/l402/purchases/direct", json=p)  

In [ ]:

pr = fs._pay_ln(ln_invoice="lnbc90n1pnk9ukupp57wcf8q2wacl5h986zch6alc24zwj8dhf42wk34dw2cuu44k5mauqdq6xysyxun9v35hggzsv93kkct8v5cqzpgxqrzpnrzjqwghf7zxvfkxq5a6sr65g0gdkv768p83mhsnt0msszapamzx2qvuxqqqqz99gpz55yqqqqqqqqqqqqqq9qrzjq25carzepgd4vqsyn44jrk85ezrpju92xyrk9apw4cdjh6yrwt5jgqqqqz99gpz55yqqqqqqqqqqqqqq9qsp564vc9ac6g6hshtxt9j4h7nh46sv9u566dlcrzug9rnvzplvgwq9s9qxpqysgqr4hhczsztdv625f6mzh2dc9u353nhtnpwcha4tp6kq2ztlqkw0ysegv6zwxj3tsfk447pyq90pszg26m9l5u9wc3xsjzrywkzvafm7gp32ap6k", description="1 credit in stock.l402.org", l402_url="https://stock.l402.org/ticker/GOOGL")
pr

<Response [500 Internal Server Error]>

In [ ]:
#| hide
# we can't auto test this because ln invoices can only be paid once
# test_eq(pr.status_code, 200)

### Pay 

In [ ]:
#| export
@patch
def pay(self:Client,
        purl:str, # payment endpoint URL
        pct:str, # payment context token
        amount:int, # amount in cents
        balance:int, # balance
        currency:str, # currency
        description:str, # description
        offer_id:str, # offer id
        payment_methods:list[str], # payment methods
        title:str, # offer title
        type:str # offer type
) -> dict: # payment status response
    "POST payment request. Returns payment status response"
    return self._request("POST", "v0/l402/purchases/from-offer", json={
        "payment_request_url": purl,
        "payment_context_token": pct,
        "offer": {
            "offer_id": offer_id,
            "title": title,
            "description": description,
            "amount": amount,
            "type": type,
            "currency": currency,
            "balance": balance,
            "payment_methods": payment_methods,
        },
    })

In [ ]:
# Example offer from stock.l402.org
ofs = {
   "offers":[
      {
         "amount":1,
         "balance":1,
         "currency":"USD",
         "description":"Purchase 1 credit for API access",
         "offer_id":"offer_c668e0c0",
         "payment_methods":[
            "lightning"
         ],
         "title":"1 Credit Package",
         "type":"top-up"
      },
      {
         "amount":100,
         "balance":120,
         "currency":"USD",
         "description":"Purchase 120 credits for API access",
         "offer_id":"offer_97bf23f7",
         "payment_methods":[
            "lightning",
            "coinbase_commerce"
         ],
         "title":"120 Credits Package",
         "type":"top-up"
      },
      {
         "amount":499,
         "balance":750,
         "currency":"USD",
         "description":"Purchase 750 credits for API access",
         "offer_id":"offer_a896b13c",
         "payment_methods":[
            "lightning",
            "coinbase_commerce",
            "credit_card"
         ],
         "title":"750 Credits Package",
         "type":"top-up"
      }
   ],
   "payment_context_token":"edb53dec-28f5-4cbb-924a-20e9003c20e1",
   "payment_request_url":"https://stock.l402.org/l402/payment-request",
   "terms_url":"https://link-to-terms.com",
   "version":"0.2.1"
}
ofs

{'offers': [{'amount': 1,
   'balance': 1,
   'currency': 'USD',
   'description': 'Purchase 1 credit for API access',
   'offer_id': 'offer_c668e0c0',
   'payment_methods': ['lightning'],
   'title': '1 Credit Package',
   'type': 'top-up'},
  {'amount': 100,
   'balance': 120,
   'currency': 'USD',
   'description': 'Purchase 120 credits for API access',
   'offer_id': 'offer_97bf23f7',
   'payment_methods': ['lightning', 'coinbase_commerce'],
   'title': '120 Credits Package',
   'type': 'top-up'},
  {'amount': 499,
   'balance': 750,
   'currency': 'USD',
   'description': 'Purchase 750 credits for API access',
   'offer_id': 'offer_a896b13c',
   'payment_methods': ['lightning', 'coinbase_commerce', 'credit_card'],
   'title': '750 Credits Package',
   'type': 'top-up'}],
 'payment_context_token': 'edb53dec-28f5-4cbb-924a-20e9003c20e1',
 'payment_request_url': 'https://stock.l402.org/l402/payment-request',
 'terms_url': 'https://link-to-terms.com',
 'version': '0.2.1'}

In [ ]:

r = fs.pay(ofs['payment_request_url'], ofs['payment_context_token'], **ofs['offers'][0])
r, r.json()

(<Response [200 OK]>,
 {'id': 200,
  'created_at': '2024-12-20T13:20:45.234Z',
  'payment_request_url': 'https://stock.l402.org/l402/payment-request',
  'payment_context_token': 'edb53dec-28f5-4cbb-924a-20e9003c20e1',
  'invoice': 'lnbc100n1pnk2mevpp5l5n5vkkuxetgkwlzuwevuf8vy644k2fay3vtn5mnnlugmlxzl5msdq6xysyxun9v35hggzsv93kkct8v5cqzpgxqrzpnrzjqwghf7zxvfkxq5a6sr65g0gdkv768p83mhsnt0msszapamzx2qvuxqqqqz99gpz55yqqqqqqqqqqqqqq9qrzjq25carzepgd4vqsyn44jrk85ezrpju92xyrk9apw4cdjh6yrwt5jgqqqqz99gpz55yqqqqqqqqqqqqqq9qsp5klt9rqrjn5h7jqx640gxrffn7a8xm3vn93p3f6lc9exmkefw0mqs9qxpqysgqzl8pl85gc2mvdh8heyjdvu0zauzye7mdvwl7nhh5kgjrpjcs4zvrf65qk2t2drgghmqyf6226uw8eet0ez77x3f3mlpyush70t2549gqg7fpxw',
  'preimage': '0a786231e8c270b95b59eff192cd42399275b6f34f0202e77bf95b5ed0e1829e',
  'amount': 1,
  'currency': 'usd',
  'payment_method': 'lightning',
  'title': '1 Credit Package',
  'description': 'Purchase 1 credit for API access',
  'type': 'top-up',
  'l402_url': '',
  'macaroon': ''})

In [ ]:

fs = Client(base_url="http://localhost:8000", api_key=os.getenv("FEWSATS_LOCAL_API_KEY"))
r = fs.pay(ofs['payment_request_url'], ofs['payment_context_token'], **ofs['offers'][0])
r, r.json()

(<Response [200 OK]>,
 {'id': 36,
  'created_at': '2024-12-20T13:20:48.182Z',
  'payment_request_url': 'https://stock.l402.org/l402/payment-request',
  'payment_context_token': 'edb53dec-28f5-4cbb-924a-20e9003c20e1',
  'invoice': 'lnbc100n1pnk2me0pp5t64fnd0a2xcu4g9e5cccyek5ac8x7rk7mhzc3599jjvgt8g5mljqdq6xysyxun9v35hggzsv93kkct8v5cqzpgxqrzpnrzjqwghf7zxvfkxq5a6sr65g0gdkv768p83mhsnt0msszapamzx2qvuxqqqqz99gpz55yqqqqqqqqqqqqqq9qrzjq25carzepgd4vqsyn44jrk85ezrpju92xyrk9apw4cdjh6yrwt5jgqqqqz99gpz55yqqqqqqqqqqqqqq9qsp5ddknjj66u7aedwp2rkta5z2tdst0whlwp9ap4wcgte8kpqpp0qmq9qxpqysgqf96urazfctfwe54ex9cqx2yzqvqhdg5vq5es0gxymevhycrzr9fku4aypll04ma42u2mqqk0sq2dq0yqpndntgmxywqneeel9d5sv4sp4nrgnc',
  'preimage': '7b7c5159834752e7921518e9467d41625682e24ba623990b9b8d2d77fd61b089',
  'amount': 1,
  'currency': 'usd',
  'payment_method': 'lightning',
  'title': '1 Credit Package',
  'description': 'Purchase 1 credit for API access',
  'type': 'top-up',
  'l402_url': '',
  'macaroon': ''})

Both the preview and purchase methods automatically use the default payment method if a charge is needed. This client provides a straightforward way to interact with the Fewsats API, making it easy for developers to integrate Fewsats functionality into their applications.

## Agent Demo

We will use [Claudette](https://claudette.answer.ai/) to demonstrate how to pay for content using the Fewsats API.

In [ ]:
from claudette import Chat, models

In [ ]:
model = models[1]
model

'claude-3-5-sonnet-20240620'

In [ ]:
fs.balance()

[{'id': 1, 'balance': 1478, 'currency': 'usd'}]

In [ ]:
chat = Chat(model, sp='You are a helpful assistant that can pay offers.', tools=[fs.pay])
pr = f"Could you pay the cheapest offer using lightning {ofs}?"
r = chat.toolloop(pr, trace_func=print)
r

Message(id='msg_01LtWLYyugqeZxFeqTatk5i6', content=[TextBlock(text='Certainly! I\'d be happy to help you pay for the cheapest offer using Lightning. Let\'s analyze the information you\'ve provided and proceed with the payment.\n\nThe cheapest offer from the given list is:\n\n- Amount: 1 cent (USD 0.01)\n- Balance: 1 credit\n- Currency: USD\n- Description: "Purchase 1 credit for API access"\n- Offer ID: offer_c668e0c0\n- Payment Method: Lightning\n- Title: "1 Credit Package"\n- Type: top-up\n\nNow, let\'s use the `pay` function to process this payment. I\'ll use the information you\'ve provided along with the details of the cheapest offer.', type='text'), ToolUseBlock(id='toolu_01EffDLd2D8ivhb7Yde2xtTF', input={'purl': 'https://stock.l402.org/l402/payment-request', 'pct': 'edb53dec-28f5-4cbb-924a-20e9003c20e1', 'amount': 1, 'balance': 1, 'currency': 'USD', 'description': 'Purchase 1 credit for API access', 'offer_id': 'offer_c668e0c0', 'payment_methods': ['lightning'], 'title': '1 Credi

Great! The payment request has been successfully submitted. The response status code 200 OK indicates that the transaction was processed successfully.

To summarize:
1. You've purchased the "1 Credit Package" for 1 cent (USD 0.01).
2. This package provides you with 1 credit for API access.
3. The payment was made using the Lightning network.

Is there anything else you'd like to know about this transaction or any other assistance you need?

<details>

- id: `msg_019BistBDJmhfrG6iZ9aQnoM`
- content: `[{'text': 'Great! The payment request has been successfully submitted. The response status code 200 OK indicates that the transaction was processed successfully.\n\nTo summarize:\n1. You\'ve purchased the "1 Credit Package" for 1 cent (USD 0.01).\n2. This package provides you with 1 credit for API access.\n3. The payment was made using the Lightning network.\n\nIs there anything else you\'d like to know about this transaction or any other assistance you need?', 'type': 'text'}]`
- model: `claude-3-5-sonnet-20240620`
- role: `assistant`
- stop_reason: `end_turn`
- stop_sequence: `None`
- type: `message`
- usage: `{'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'input_tokens': 1402, 'output_tokens': 108}`

</details>

We can see in the chat history to see that the agent correctlye filled the required information for the payment.

The payment balance has also decreased as expected.

In [ ]:
fs.balance(), chat.h

([{'id': 1, 'balance': 1477, 'currency': 'usd'}],
 [{'role': 'user',
   'content': [{'type': 'text',
     'text': "Could you pay the cheapest offer using lightning {'offers': [{'amount': 1, 'balance': 1, 'currency': 'USD', 'description': 'Purchase 1 credit for API access', 'offer_id': 'offer_c668e0c0', 'payment_methods': ['lightning'], 'title': '1 Credit Package', 'type': 'top-up'}, {'amount': 100, 'balance': 120, 'currency': 'USD', 'description': 'Purchase 120 credits for API access', 'offer_id': 'offer_97bf23f7', 'payment_methods': ['lightning', 'coinbase_commerce'], 'title': '120 Credits Package', 'type': 'top-up'}, {'amount': 499, 'balance': 750, 'currency': 'USD', 'description': 'Purchase 750 credits for API access', 'offer_id': 'offer_a896b13c', 'payment_methods': ['lightning', 'coinbase_commerce', 'credit_card'], 'title': '750 Credits Package', 'type': 'top-up'}], 'payment_context_token': 'edb53dec-28f5-4cbb-924a-20e9003c20e1', 'payment_request_url': 'https://stock.l402.org/l402

In [ ]:
#|hide
from nbdev.doclinks import nbdev_export
nbdev_export()